In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np

from src.data import load_train, load_test

train = load_train()
test = load_test()

print("Training shape:", train.shape)
print("Testing shape:", test.shape)

Training shape: (175341, 45)
Testing shape: (82332, 45)


In [2]:
excluded_columns = ["id", "attack_cat", "label"]

feature_columns = [
    col for col in train.columns
    if col not in excluded_columns
]

X_train = train[feature_columns]
y_train = train["label"]

X_test = test[feature_columns]
y_test = test["label"]

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

X_train shape: (175341, 42)
y_train shape: (175341,)
X_test shape: (82332, 42)
y_test shape: (82332,)


In [3]:
# Identify categorical columns
categorical_columns = X_train.select_dtypes(
    include=["object", "category"]
).columns.tolist()

# Identify numerical columns
numerical_columns = X_train.select_dtypes(
    include=["number"]
).columns.tolist()

print("Categorical columns:")
print(categorical_columns)

print("\nNumber of categorical features:", len(categorical_columns))

print("\nNumber of numerical features:", len(numerical_columns))
print("\nNumerical columns:")
print(numerical_columns)

print("\nTotal features:", len(categorical_columns) + len(numerical_columns))

Categorical columns:
['proto', 'service', 'state']

Number of categorical features: 3

Number of numerical features: 39

Numerical columns:
['dur', 'spkts', 'dpkts', 'sbytes', 'dbytes', 'rate', 'sttl', 'dttl', 'sload', 'dload', 'sloss', 'dloss', 'sinpkt', 'dinpkt', 'sjit', 'djit', 'swin', 'stcpb', 'dtcpb', 'dwin', 'tcprtt', 'synack', 'ackdat', 'smean', 'dmean', 'trans_depth', 'response_body_len', 'ct_srv_src', 'ct_state_ttl', 'ct_dst_ltm', 'ct_src_dport_ltm', 'ct_dst_sport_ltm', 'ct_dst_src_ltm', 'is_ftp_login', 'ct_ftp_cmd', 'ct_flw_http_mthd', 'ct_src_ltm', 'ct_srv_dst', 'is_sm_ips_ports']

Total features: 42


In [4]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Numerical preprocessing
numeric_transformer = StandardScaler()

# Categorical preprocessing
categorical_transformer = OneHotEncoder(
    handle_unknown="ignore"
)

# Combine both preprocessing methods
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numerical_columns),
        ("cat", categorical_transformer, categorical_columns)
    ]
)

print("Preprocessor created successfully!")

Preprocessor created successfully!


In [5]:
# Learn preprocessing rules from training data
X_train_processed = preprocessor.fit_transform(X_train)

print("Training data before preprocessing:", X_train.shape)
print("Training data after preprocessing:", X_train_processed.shape)

Training data before preprocessing: (175341, 42)
Training data after preprocessing: (175341, 194)


In [6]:
# Apply the already-fitted preprocessor to testing data
X_test_processed = preprocessor.transform(X_test)

print("Testing data before preprocessing:", X_test.shape)
print("Testing data after preprocessing:", X_test_processed.shape)

Testing data before preprocessing: (82332, 42)
Testing data after preprocessing: (82332, 194)


In [8]:
from scipy import sparse

def count_missing(matrix):
    if sparse.issparse(matrix):
        return np.isnan(matrix.data).sum()
    return np.isnan(matrix).sum()

print("Training processed shape:", X_train_processed.shape)
print("Testing processed shape:", X_test_processed.shape)

print("\nMissing values in training:",
      count_missing(X_train_processed))

print("Missing values in testing:",
      count_missing(X_test_processed))

Training processed shape: (175341, 194)
Testing processed shape: (82332, 194)

Missing values in training: 0
Missing values in testing: 0


In [9]:
from src.preprocessing import create_preprocessor

# Create a new shared preprocessor
shared_preprocessor = create_preprocessor(numerical_columns)

# Fit using training data only
X_train_shared = shared_preprocessor.fit_transform(X_train)

# Transform testing data using the same fitted preprocessor
X_test_shared = shared_preprocessor.transform(X_test)

print("Training shape:", X_train_shared.shape)
print("Testing shape:", X_test_shared.shape)

Training shape: (175341, 194)
Testing shape: (82332, 194)
